# `PDBX` — profile, choose columns & trim

Trim type: **row**. Choose **6 columns**, **1,500,000 rows**. Save `PDBX_6c_1500000r.csv`.

> Output columns: `c0`…`c5`.

In [ ]:
import os
import numpy as np
import pandas as pd

NAME       = "PDBX"
N_ROWS     = 17305799
N_COLS     = 6
TRIM_ROWS  = "head"   # head | sample

RAW_PATH   = "PDBX_POLY_SEQ_SCHEME_r17305799_c13.csv"
OUT_PATH   = f"{NAME}_{N_COLS}c_{N_ROWS}r.csv"
DELIM      = ","
OUT_DELIM  = DELIM
HAS_HEADER = False
ENCODING   = "utf-8"
ON_BAD_LINES = None

## 1. View the raw data

In [4]:
read_kw = dict(sep=DELIM, header=0 if HAS_HEADER else None, encoding=ENCODING, low_memory=False)
if ON_BAD_LINES is not None:
    read_kw["on_bad_lines"] = ON_BAD_LINES
raw = pd.read_csv(RAW_PATH, **read_kw)
raw.columns = [f"c{i}" for i in range(raw.shape[1])]
print("raw shape :", raw.shape)
print("columns   :", list(raw.columns))
raw.head()

raw shape : (17305799, 13)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11', 'c12']


,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9,c10,c11,c12
0,195116,326,C,3,125,LYS,A,125,299,299,LYS,LYS,.
1,195117,326,C,3,126,CYS,A,126,300,300,CYS,CYS,.
2,195118,326,C,3,127,ASP,A,127,301,301,ASP,ASP,.
3,195119,326,C,3,128,PHE,A,128,302,302,PHE,PHE,.
4,195120,326,C,3,129,THR,A,129,303,303,THR,THR,.


In [ ]:
raw.dtypes

## 2. Profile: cardinality, top-value %, and group skew

In [5]:
total_rows = len(raw)

rows = []
for col in raw.columns:
    vc = raw[col].value_counts(dropna=False)
    distinct = int(raw[col].nunique(dropna=True))
    nulls    = int(raw[col].isna().sum() + (raw[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))

prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")

prof

,distinct_values,null_count,null_percent,unique_percent,top_value_percent,mean_group,max_group,skew_ratio
column,,,,,,,,
c0,17305799,0,0.00,100.00,0.00,1.00,1,1.00
c1,32462,0,0.00,0.19,0.09,533.11,15696,29.44
c2,67,1176,0.01,0.00,44.63,254497.04,7722915,30.35
c3,45,0,0.00,0.00,83.52,384573.31,14454154,37.58
c4,2999,0,0.00,0.02,0.45,5770.52,77121,13.36
c5,896,0,0.00,0.01,8.56,19314.51,1480812,76.67
c6,62,0,0.00,0.00,43.72,279125.79,7566189,27.11
c7,2999,0,0.00,0.02,0.45,5770.52,77121,13.36
c8,9830,0,0.00,0.06,0.34,1760.51,58711,33.35


## 3. Choose columns (cardinality mix + id/super-key)

In [6]:
prof_sel = prof[prof.distinct_values > 1]
KEY_THRESH = 95
keys     = prof_sel[prof_sel.unique_percent >= KEY_THRESH]
non_keys = prof_sel[prof_sel.unique_percent <  KEY_THRESH]
chosen = []
if len(keys):
    chosen.append(keys.sort_values("unique_percent", ascending=False).index[0])
n_left = N_COLS - len(chosen)
nk     = non_keys.sort_values("unique_percent")
ix_strata = np.array_split(nk.index.to_numpy(), min(3, max(1, len(nk)))) if len(nk) else []
strata = [nk.loc[ix].sort_values("skew_ratio", ascending=False) for ix in ix_strata]
ptr    = [0] * len(strata)
while n_left > 0 and any(ptr[i] < len(strata[i]) for i in range(len(strata))):
    for i in range(len(strata)):
        if ptr[i] < len(strata[i]):
            chosen.append(strata[i].index[ptr[i]]); ptr[i] += 1; n_left -= 1
            if n_left == 0: break
for c in raw.columns:
    if len(chosen) >= N_COLS: break
    if c not in chosen: chosen.append(c)
SELECTED_COLS = [c for c in raw.columns if c in set(chosen)][:N_COLS]
assert len(SELECTED_COLS) == N_COLS, SELECTED_COLS
print("selected columns:", SELECTED_COLS)

selected columns: ['c0', 'c3', 'c5', 'c9', 'c10', 'c11']


## 4. Trim to the chosen columns x exact rows

In [7]:
assert raw.shape[1] >= N_COLS, f"need >= {N_COLS} cols, have {raw.shape[1]}"
n_take = min(N_ROWS, len(raw))
assert len(raw) >= n_take, f"need >= {n_take} rows, have {len(raw)}"

if TRIM_ROWS == "sample":
    trimmed = raw.loc[:, SELECTED_COLS].sample(n_take, random_state=42).reset_index(drop=True)
else:
    trimmed = raw.loc[:, SELECTED_COLS].iloc[:n_take].copy()
trimmed.columns = [f"c{i}" for i in range(N_COLS)]

assert trimmed.shape == (n_take, N_COLS), trimmed.shape
print("trimmed shape:", trimmed.shape)
trimmed.head()

trimmed shape: (17305799, 6)


,c0,c1,c2,c3,c4,c5
0,195116,3,LYS,299,LYS,LYS
1,195117,3,CYS,300,CYS,CYS
2,195118,3,ASP,301,ASP,ASP
3,195119,3,PHE,302,PHE,PHE
4,195120,3,THR,303,THR,THR


## 5. Save the trimmed CSV

In [8]:
out_dir = os.path.dirname(OUT_PATH)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
trimmed.to_csv(OUT_PATH, index=False, sep=OUT_DELIM)
print("wrote", OUT_PATH, trimmed.shape)

chk = pd.read_csv(OUT_PATH, sep=OUT_DELIM)
print("reloaded:", chk.shape)
print("columns   :", list(chk.columns))

wrote PDBX_6c_17305799r.csv (17305799, 6)
reloaded: (17305799, 6)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5']


## 6. Check selected cardinality and skew

In [ ]:
output = pd.read_csv(OUT_PATH, sep=OUT_DELIM, encoding=ENCODING, low_memory=False)
print("output shape :", output.shape)
print("columns      :", list(output.columns))
output.head()

In [ ]:
total_rows = len(output)
rows = []
for col in output.columns:
    vc = output[col].value_counts(dropna=False)
    distinct = int(output[col].nunique(dropna=True))
    nulls    = int(output[col].isna().sum() + (output[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))
out_prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")
out_prof